# Bolzano (EURAC, Trina)

## Paths

In [ ]:
from pathlib import Path

ROOT = Path("../..").resolve()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Locate raw data

In [ ]:
import os
import pandas as pd

EURAC_DIR = str(RAW_DIR / 'EURAC')
TRINA_PATH = os.path.join(EURAC_DIR, 'Trina', 'Trina.csv')

if not os.path.exists(EURAC_DIR):
    raise FileNotFoundError(f"Folder not found: {EURAC_DIR}")

if not os.path.exists(TRINA_PATH):
    raise FileNotFoundError(f"File not found: {TRINA_PATH}")

## Build unified hourly dataset

In [ ]:
import numpy as np

df_raw = pd.read_csv(TRINA_PATH, sep=';', on_bad_lines='skip', engine='python')

df_raw['Fecha'] = pd.to_datetime(df_raw['Fecha']).dt.tz_localize(None)
df_raw.set_index('Fecha', inplace=True)

if 'Pmpp (W)' in df_raw.columns and df_raw['Pmpp (W)'].sum() > 0:
    df_raw['Power_W'] = df_raw['Pmpp (W)']
elif {'Vmpp(V)', 'Impp(A)'}.issubset(df_raw.columns):
    df_raw['Power_W'] = df_raw['Vmpp(V)'] * df_raw['Impp(A)']
else:
    df_raw['Power_W'] = 0.0

if df_raw['Power_W'].sum() == 0 and 'Pac(W)' in df_raw.columns:
    df_raw['Power_W'] = df_raw['Pac(W)']

df_hourly = df_raw.resample('1h').mean().reset_index()

df_eurac_trina = pd.DataFrame()
df_eurac_trina['utctime'] = df_hourly['Fecha']
df_eurac_trina['Power_Actual_W'] = df_hourly['Power_W'].fillna(0.0)

P_MAX = df_eurac_trina['Power_Actual_W'].max()
if pd.isna(P_MAX) or P_MAX <= 0:
    P_MAX = 1.0
df_eurac_trina['Power_PU'] = np.clip(df_eurac_trina['Power_Actual_W'] / P_MAX, 0.0, 1.0)

df_eurac_trina['Irradiance_POA'] = df_hourly['Front GPOA (W/m²)'].clip(lower=0.0).fillna(0.0)
df_eurac_trina['GHI'] = df_hourly['GHI (W/m²)'].clip(lower=0.0).fillna(0.0)
df_eurac_trina['Temp_Ambient'] = df_hourly['Amb. Temp. (°C)'].fillna(df_hourly['Amb. Temp. (°C)'].mean())
df_eurac_trina['Wind_Speed'] = df_hourly['Wind Speed (m/s)'].clip(lower=0.0).fillna(0.0)
df_eurac_trina['Humidity'] = 50.0

hours = df_eurac_trina['utctime'].dt.hour
months = df_eurac_trina['utctime'].dt.month
df_eurac_trina['HOUR_SIN'] = np.sin(2 * np.pi * hours / 24.0)
df_eurac_trina['HOUR_COS'] = np.cos(2 * np.pi * hours / 24.0)
df_eurac_trina['MONTH_SIN'] = np.sin(2 * np.pi * months / 12.0)
df_eurac_trina['MONTH_COS'] = np.cos(2 * np.pi * months / 12.0)

LATITUDE, LONGITUDE, TILT = 46.45761, 11.3285, 30.0
df_eurac_trina['Latitude'] = LATITUDE
df_eurac_trina['Longitude'] = LONGITUDE
df_eurac_trina['Tilt'] = TILT

FINAL_COLUMNS = [
    'utctime', 'Power_Actual_W', 'Power_PU',
    'Irradiance_POA', 'GHI', 'Temp_Ambient', 'Wind_Speed', 'Humidity',
    'HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS',
    'Latitude', 'Longitude', 'Tilt'
]
df_eurac_trina = df_eurac_trina[FINAL_COLUMNS]

## Relative humidity (NASA POWER)

In [ ]:
import requests
import io

start_date, end_date = "20150101", "20151231"

nasa_url = (
    "https://power.larc.nasa.gov/api/temporal/hourly/point?"
    f"parameters=RH2M&community=RE&longitude={LONGITUDE}&latitude={LATITUDE}&"
    f"start={start_date}&end={end_date}&format=CSV"
)

response = requests.get(nasa_url)

if response.status_code != 200:
    raise RuntimeError(f"NASA POWER request failed: {response.status_code}")

decoded_content = response.content.decode('utf-8')

lines = decoded_content.split('\n')
data_start_idx = next((i for i, line in enumerate(lines) if line.startswith('YEAR')), 0)

df_nasa = pd.read_csv(io.StringIO(decoded_content), skiprows=data_start_idx)

df_nasa['utctime'] = pd.to_datetime(
    df_nasa['YEAR'].astype(str) + '-' +
    df_nasa['MO'].astype(str).str.zfill(2) + '-' +
    df_nasa['DY'].astype(str).str.zfill(2) + ' ' +
    df_nasa['HR'].astype(str).str.zfill(2) + ':00:00'
)

df_nasa['Humidity'] = df_nasa['RH2M'].replace([-999, -999.0], pd.NA)
df_nasa['Humidity'] = df_nasa['Humidity'].interpolate(method='linear').ffill().bfill()

df_humidity_fixed = df_nasa[['utctime', 'Humidity']].copy()

df_eurac_trina = df_eurac_trina.drop(columns=['Humidity'], errors='ignore')
df_eurac_trina = pd.merge(df_eurac_trina, df_humidity_fixed, on='utctime', how='left')
df_eurac_trina['Humidity'] = df_eurac_trina['Humidity'].interpolate(method='linear').ffill().bfill()

## Checks

In [ ]:
EXPECTED_ROWS = 8760

assert len(df_eurac_trina) == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS} rows, got {len(df_eurac_trina)}"
)
assert df_eurac_trina['utctime'].is_monotonic_increasing, "Timestamps are not sorted"

## Save

In [ ]:
SAVE_PATH = OUT_DIR / 'Bolzano.csv'
df_eurac_trina.to_csv(SAVE_PATH, index=False)